In [1]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

In [2]:
raw_dir = "../data/raw/variant_04/"
print("Файлы в папке raw:", os.listdir(raw_dir))

# Замените имя файла на то, что вы увидите в выводе выше
json_filename = "2024-01-01_2024-01-07_2026-05-28_14-41-18.json"

full_path = os.path.join(raw_dir, json_filename)

if os.path.exists(full_path):
    print(" Файл найден:", full_path)
else:
    print(" Файл не найден! Проверьте имя.")

Файлы в папке raw: ['.gitkeep', '2024-01-01_2024-01-07_2026-05-28_14-41-18.json']
 Файл найден: ../data/raw/variant_04/2024-01-01_2024-01-07_2026-05-28_14-41-18.json


In [3]:
with open(full_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print("Тип данных:", type(data))
print("Ключи верхнего уровня:", list(data.keys()))

if 'hourly' in data:
    print("Ключи внутри hourly:", list(data['hourly'].keys()))
    print("Количество временных меток:", len(data['hourly']['time']))

Тип данных: <class 'dict'>
Ключи верхнего уровня: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Ключи внутри hourly: ['time', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_speed_10m']
Количество временных меток: 168


In [4]:
df = pd.DataFrame({
    'timestamp': data['hourly']['time'],
    'temperature': data['hourly']['temperature_2m'],
    'humidity': data['hourly']['relative_humidity_2m'],
    'precipitation': data['hourly']['precipitation'],
    'wind_speed': data['hourly']['wind_speed_10m']
})

df['city'] = 'London'
df['variant_id'] = '04'

print("Первые 3 строки:")
print(df.head(3))
print("\nРазмер таблицы:", df.shape)
print("\nТипы данных ДО очистки:")
print(df.dtypes)

Первые 3 строки:
          timestamp  temperature  humidity  precipitation  wind_speed    city  \
0  2024-01-01T00:00          7.7        77            0.0        30.6  London   
1  2024-01-01T01:00          7.6        77            0.0        29.5  London   
2  2024-01-01T02:00          7.3        78            0.0        29.0  London   

  variant_id  
0         04  
1         04  
2         04  

Размер таблицы: (168, 7)

Типы данных ДО очистки:
timestamp         object
temperature      float64
humidity           int64
precipitation    float64
wind_speed       float64
city              object
variant_id        object
dtype: object


In [5]:
# 1) Преобразуем timestamp в datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2) Проверяем пропуски
print("Пропуски:\n", df.isnull().sum())
# Если есть пропуски в осадках – заменим на 0
if df['precipitation'].isnull().sum() > 0:
    df['precipitation'] = df['precipitation'].fillna(0)

# 3) Проверяем дубликаты
print("Дубликаты строк:", df.duplicated().sum())
# Если есть – удалим (сейчас их нет, но код не повредит)
df = df.drop_duplicates()

Пропуски:
 timestamp        0
temperature      0
humidity         0
precipitation    0
wind_speed       0
city             0
variant_id       0
dtype: int64
Дубликаты строк: 0


In [6]:
df = df.rename(columns={
    'timestamp': 'datetime_utc',
    'temperature': 'temp_c',
    'humidity': 'rel_humidity_percent',
    'precipitation': 'precip_mm',
    'wind_speed': 'wind_kmh'
})
print("Новые колонки:", list(df.columns))

Новые колонки: ['datetime_utc', 'temp_c', 'rel_humidity_percent', 'precip_mm', 'wind_kmh', 'city', 'variant_id']


In [7]:
print("=== Диапазоны ===")
print("Температура, °C:", df['temp_c'].min(), "–", df['temp_c'].max())
print("Влажность, %:", df['rel_humidity_percent'].min(), "–", df['rel_humidity_percent'].max())
print("Осадки, мм:", df['precip_mm'].min(), "–", df['precip_mm'].max())
print("Ветер, км/ч:", df['wind_kmh'].min(), "–", df['wind_kmh'].max())

=== Диапазоны ===
Температура, °C: 1.5 – 12.9
Влажность, %: 60 – 98
Осадки, мм: 0.0 – 8.0
Ветер, км/ч: 2.8 – 52.3


In [8]:
normalized_dir = "../data/normalized/variant_04"
os.makedirs(normalized_dir, exist_ok=True)

now_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
csv_path = os.path.join(normalized_dir, f"{now_str}.csv")

df.to_csv(csv_path, index=False, encoding='utf-8')
print(f" CSV сохранён: {csv_path}")
print(f"Размер: {df.shape[0]} строк, {df.shape[1]} столбцов")

 CSV сохранён: ../data/normalized/variant_04\2026-05-28_16-59-43.csv
Размер: 168 строк, 7 столбцов


In [9]:
print("Итоговая информация:")
df.info()
print("\nПервые 2 строки:")
print(df.head(2))

Итоговая информация:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime_utc          168 non-null    datetime64[ns]
 1   temp_c                168 non-null    float64       
 2   rel_humidity_percent  168 non-null    int64         
 3   precip_mm             168 non-null    float64       
 4   wind_kmh              168 non-null    float64       
 5   city                  168 non-null    object        
 6   variant_id            168 non-null    object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(2)
memory usage: 9.3+ KB

Первые 2 строки:
         datetime_utc  temp_c  rel_humidity_percent  precip_mm  wind_kmh  \
0 2024-01-01 00:00:00     7.7                    77        0.0      30.6   
1 2024-01-01 01:00:00     7.6                    77        0.0      29.5   

     city variant_id  
0 